# 23 — Choosing: Plain API Management vs. AI Gateway Policies

Notebooks 16-22 showed you *how* to expose, secure, govern, and call MCP
servers and LLM backends through Azure API Management. This notebook
answers a different question: **when should you reach for an AI Gateway
policy versus a plain, ordinary API Management policy you'd use for any
REST API?**

This matters because "just enable AI Gateway" isn't a real decision --
AI Gateway isn't a switch, it's a set of specific policies
(`llm-token-limit`, `llm-semantic-cache-lookup`/`-store`,
`llm-content-safety`, and friends) layered on the API gateway you already
have. Every API Management instance already *has* an API gateway; the
question is only which policies a given API actually needs.

Read alongside [docs/11_apim_vs_ai_gateway_decision_guide.md](../docs/11_apim_vs_ai_gateway_decision_guide.md)
and try the recipes in
[examples/apim_policies/](../examples/apim_policies/README.md).

## The core distinction

| | Plain API Management policies | AI Gateway policies |
| --- | --- | --- |
| Governs by | Call count, bandwidth, concurrency | LLM **tokens**, semantic **similarity**, content **harm categories** |
| Examples | `rate-limit-by-key`, `quota-by-key`, `cache-lookup`/`cache-store`, `ip-filter`, `validate-azure-ad-token` | `llm-token-limit`, `llm-emit-token-metric`, `llm-semantic-cache-lookup`/`-store`, `llm-content-safety` |
| Applies to | Any REST API, including MCP servers' tool-call traffic | LLM chat/completions/embeddings APIs (OpenAI, Azure OpenAI, Anthropic, Vertex AI schemas) |
| Extra prerequisites | None beyond API Management itself | Often needs an embeddings deployment, a Content Safety resource, or Application Insights wired up |

Both families are official, first-class Azure API Management policies --
"AI Gateway" is Microsoft's name for the *category*, in the same policy
reference page as `rate-limit` and `cache-lookup`
([api-management-policies](https://learn.microsoft.com/en-us/azure/api-management/api-management-policies#ai-gateway)).

## Decision table: signal → policy family

| If you're seeing... | Reach for... | Family |
| --- | --- | --- |
| A client hammering an endpoint with request volume | `rate-limit-by-key` / `quota-by-key` | Plain |
| One request silently costing 50x another (long prompts) | `llm-token-limit` / `azure-openai-token-limit` | AI Gateway |
| Repeated identical requests wasting backend capacity | `cache-lookup` / `cache-store` | Plain |
| Repeated *differently worded* requests with the same meaning | `llm-semantic-cache-lookup` / `-store` | AI Gateway |
| No visibility into who's driving cost | `emit-metric` (generic) | Plain |
| No visibility into who's driving cost, and cost = tokens | `llm-emit-token-metric` | AI Gateway |
| Untrusted users submitting free-text prompts | `llm-content-safety` | AI Gateway |
| A single backend that's a single point of failure | backend pool + circuit breaker | Both -- resiliency isn't AI-specific, but LLM backends benefit the most (notebook 21) |
| Callers need to be authenticated | `validate-azure-ad-token` / `validate-jwt` | Plain -- identical whether the backend is an LLM or not |

## A five-question decision tree

1. **Does the backend consume or produce natural-language tokens** (a chat/completions
   API, an embeddings API, or an MCP tool whose result feeds an LLM)?
   - **No** → you almost certainly only need plain API Management policies.
     Stop here.
   - **Yes** → continue.
2. **Do you need per-caller cost or quota control finer than "N calls"?**
   → `llm-token-limit` / `azure-openai-token-limit`
   ([examples/apim_policies/ai_gateway/01](../examples/apim_policies/ai_gateway/01_llm_token_limit.xml)).
3. **Do near-duplicate prompts show up often enough that caching by meaning
   would help?** → `llm-semantic-cache-lookup` / `-store`
   ([ai_gateway/06](../examples/apim_policies/ai_gateway/06_llm_semantic_cache_lookup_and_store.xml)).
4. **Could a prompt or response contain harmful content you must block?**
   → `llm-content-safety`
   ([ai_gateway/09](../examples/apim_policies/ai_gateway/09_llm_content_safety.xml)).
5. **Do you have (or want) more than one model deployment?** → a backend
   pool with load balancing and a circuit breaker
   ([ai_gateway/11-15](../examples/apim_policies/ai_gateway/)).

Answering "no" to 2-5 doesn't mean you made a mistake -- it means AI
Gateway's *extra* capabilities don't apply to your scenario yet, and plain
API Management policies (auth, plain rate limiting, tracing) are still
exactly what you should use.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

def recommend_policies(
    *,
    backend_is_llm: bool,
    needs_token_cost_control: bool = False,
    repeats_similar_prompts: bool = False,
    needs_content_safety: bool = False,
    has_multiple_deployments: bool = False,
) -> list[str]:
    '''A tiny, literal version of the decision tree above -- point this at
    your own scenario and it returns which example files to start from.'''
    if not backend_is_llm:
        return [
            "examples/apim_policies/plain/01_rate_limit_by_key.xml",
            "examples/apim_policies/plain/09_validate_azure_ad_token.xml",
        ]

    recipes = []
    if needs_token_cost_control:
        recipes.append("examples/apim_policies/ai_gateway/01_llm_token_limit.xml")
    if repeats_similar_prompts:
        recipes.append("examples/apim_policies/ai_gateway/06_llm_semantic_cache_lookup_and_store.xml")
    if needs_content_safety:
        recipes.append("examples/apim_policies/ai_gateway/09_llm_content_safety.xml")
    if has_multiple_deployments:
        recipes.append("examples/apim_policies/ai_gateway/13_backend_load_balancing_priority_ptu.bicep")
    if not recipes:
        recipes.append("examples/apim_policies/ai_gateway/04_llm_emit_token_metric.xml")
    return recipes

# Try it against the "internal chatbot" scenario from later in this notebook:
recommend_policies(
    backend_is_llm=True,
    needs_token_cost_control=True,
    repeats_similar_prompts=False,
    needs_content_safety=True,
    has_multiple_deployments=False,
)

## Cost and complexity, not just capability

AI Gateway policies aren't free to turn on:

- `llm-semantic-cache-lookup`/`-store` need an embeddings deployment
  ([ai_gateway/08](../examples/apim_policies/ai_gateway/08_semantic_cache_embeddings_backend.bicep))
  and, per the policy reference, an external cache -- that's infrastructure
  you now operate and pay for.
- `llm-content-safety` needs an Azure AI Content Safety resource and a
  backend configured with the right managed-identity role.
- `llm-token-limit`, `llm-emit-token-metric`, and `llm-semantic-cache-store`
  are **not** available on the Consumption tier -- check the policy
  reference's tier columns before you design around one of them.
- Every AI Gateway policy still benefits from -- and doesn't replace --
  ordinary auth and network policies (notebook 18).

Don't add an AI Gateway policy speculatively "because this is an AI
project." Add it when the decision tree above gives you a concrete yes.

## Worked example 1: adding one LLM feature to an existing API Management instance

**Situation:** You already run API Management in front of a dozen REST
microservices, each with its own `rate-limit-by-key` and
`validate-azure-ad-token` policies. Product added one new feature backed by
Azure OpenAI.

**Recommended shape:**

- Import the Azure OpenAI API as its own API in the *same* instance --
  don't stand up a second gateway.
- Leave the existing 12 APIs' policies untouched.
- Add `llm-token-limit` and `llm-emit-token-metric` only to the new API.
- Reuse the instance's existing Entra ID app registration and
  `validate-azure-ad-token` policy for auth -- that part doesn't change.

This is the common case, and it's why AI Gateway is a *capability*, not a
separate product: you're extending governance you already trust, not
starting over.

## Worked example 2: a single internal chatbot, no existing API Management footprint

**Situation:** One team wants a Slack bot that calls Azure OpenAI. No other
APIs exist yet.

**Ask first:** do you need API Management at all yet? Azure OpenAI already
has its own quota/rate limits per deployment. For a single internal
consumer with no governance requirements beyond "don't blow the budget,"
calling Azure OpenAI directly (as in notebooks 06-07, Part 1 of this
course) may be simpler and cheaper than standing up API Management.

**What tips the balance toward introducing API Management + AI Gateway
anyway:**

- More than one consumer/team will need access, with different budgets.
- You need centralized observability (token metrics per team) that Azure
  OpenAI alone doesn't give you across consumers.
- You also want to expose MCP tools (notebooks 17/20) alongside the model --
  once you need governed MCP servers, you already have the gateway, so
  fronting the LLM traffic through it too is close to free.

In [ ]:
from src.config import get_settings

settings = get_settings()
print("APIM MCP server configured:", settings.has_apim_mcp)
print("APIM AI Gateway (LLM) endpoint configured:", bool(settings.apim_gateway_endpoint))

if settings.has_apim_mcp and settings.apim_gateway_endpoint:
    print("Your setup matches worked example 1: one gateway fronting both MCP tools and an LLM backend.")
elif settings.apim_gateway_endpoint and not settings.has_apim_mcp:
    print("Your setup matches worked example 2 so far: LLM traffic through the gateway, no MCP server yet.")
else:
    print("Neither is configured yet -- revisit notebooks 16/17/21 to set one up before running this cell for real.")

## Recap and checklist

- [ ] I can name which *specific* policy I need, not just "turn on AI Gateway."
- [ ] Every AI Gateway policy I plan to use has its prerequisite
      infrastructure (embeddings deployment, Content Safety resource,
      Application Insights) accounted for.
- [ ] Auth, network, and generic observability policies stay the plain
      ones from notebook 18/19 -- I only added AI-specific policies where
      the decision tree said yes.
- [ ] I'm reusing one API Management instance for both existing REST APIs
      and new LLM/MCP traffic where reasonable, per worked example 1.

This closes out the AI Gateway arc of the course. From here:

- [docs/11_apim_vs_ai_gateway_decision_guide.md](../docs/11_apim_vs_ai_gateway_decision_guide.md) --
  the conceptual companion to this notebook.
- [examples/apim_policies/](../examples/apim_policies/README.md) -- all 30
  recipes referenced above, ready to adapt.
- [Azure-Samples/AI-Gateway](https://github.com/Azure-Samples/AI-Gateway) --
  end-to-end labs combining these policies in real deployments.